In [1]:
import pandas as pd 
import numpy as np 

**1. Create time indice: read every 15 min for 30 days**

In [12]:
dates = pd.date_range(start='2026-03-01', end='2026-03-30', freq='15min')
n = len(dates)

In [13]:
np.random.seed(42)
air_temp = 25 + 10 * np.sin(np.linspace(0, 30*2*np.pi, n)) + np.random.normal(0, 1, n)
soil_umid = 40 - 5 * np.cos(np.linspace(0, 30*2*np.pi, n)) + np.random.normal(0, 2, n)
solar_rad = np.where(np.sin(np.linspace(0, 30*2*np.pi, n)) > 0, 
                     800 * np.sin(np.linspace(0, 30*2*np.pi, n)), 0) + np.random.normal(0, 50, n)
solar_rad = np.clip(solar_rad, 0, 1200)

In [14]:
df_raw = pd.DataFrame({
    'timestamp': dates,
    'air_temp_c': air_temp,
    'soil_umid_pct': soil_umid,
    'solar_rad_w_m2': solar_rad
})

In [41]:
#Fails injection (NaN, outliers)
df_raw.loc[500:550, 'soil_umid_pct'] = np.nan
df_raw.loc[1200:1220, ['air_temp_c', 'soil_umid_pct']] = np.nan
df_raw.loc[800, 'air_temp_c'] = 85.5
df_raw.loc[1500, 'solar_rad_w_m2'] = -15.7

df_raw.to_csv('iot_agri_raw.csv', index=False)
print("Raw data saved to 'iot_agri_raw.csv'")

Raw data saved to 'iot_agri_raw.csv'


In [42]:
df = pd.read_csv('iot_agri_raw.csv')
display(df.head())
df.info()

,timestamp,air_temp_c,soil_umid_pct,solar_rad_w_m2
0,2026-03-01 00:00:00,25.496714,34.296157,0.000000
1,2026-03-01 00:15:00,25.538286,33.471464,27.469496
2,2026-03-01 00:30:00,26.997689,32.453539,90.796813
3,2026-03-01 00:45:00,28.540294,34.190549,142.654394
4,2026-03-01 01:00:00,27.441130,35.545103,165.026392


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2785 entries, 0 to 2784
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   timestamp       2785 non-null   object 
 1   air_temp_c      2764 non-null   float64
 2   soil_umid_pct   2713 non-null   float64
 3   solar_rad_w_m2  2785 non-null   float64
dtypes: float64(3), object(1)
memory usage: 87.2+ KB


In [43]:
df['timestamp'] = pd.to_datetime(df['timestamp'])
df.set_index('timestamp', inplace=True)
print("Missing data by sensor:")
display(df.isna().sum())

Missing data by sensor:


air_temp_c        21
soil_umid_pct     72
solar_rad_w_m2     0
dtype: int64

**2. Data Treatment with Linear Interpolation**

In [44]:
#Fill the gaps (NaN) with linear interpolation
df['air_temp_c'] = df['air_temp_c'].interpolate(method='linear')
df['soil_umid_pct'] = df['soil_umid_pct'].interpolate(method='linear')

#Check if interpolation worked
print("Missing data after interpolation:")
display(df.isna().sum())

Missing data after interpolation:


air_temp_c        0
soil_umid_pct     0
solar_rad_w_m2    0
dtype: int64

**3. Simple Outliers Hunt**

In [45]:
# 1. Define columns to analyze for anomalies
sensor_columns = ['air_temp_c', 'soil_umid_pct']

# 2. Loop through each sensor column
for column in sensor_columns:
    
    mean = df[column].mean()
    std = df[column].std()
    
    sup_threshold = mean + (3 * std)
    inf_threshold = mean - (3 * std)
    
    # Identifying anomalies (Outliers)
    anomalies = df[(df[column] > sup_threshold) | (df[column] < inf_threshold)]
    
    print(f"\n=== REPORT: {column.upper()} ===")
    print(f"Acceptable Limits: {inf_threshold:.2f} to {sup_threshold:.2f}")
    print(f"Anomalies found: {len(anomalies)}")
    
    # 3. Transform to NaN and Interpolate
    if len(anomalies) > 0:
        df.loc[df[column] > sup_threshold, column] = np.nan
        df.loc[df[column] < inf_threshold, column] = np.nan
        
        # Linear Interpolation to close the gap
        df[column] = df[column].interpolate(method='linear')
        
        anom_position = df.index.get_loc(anomalies.index[0])

        print(f"[!] Automatic correction applied to {column}.")

        print(f"Missing data after outlier treatment:")
        display(df.iloc[anom_position - 1 : anom_position + 2][[column]])
    else:
        print("[v] No critical errors found.")


=== REPORT: AIR_TEMP_C ===
Acceptable Limits: 3.31 to 46.82
Anomalies found: 1
[!] Automatic correction applied to air_temp_c.
Missing data after outlier treatment:


,air_temp_c
timestamp,
2026-03-09 07:45:00,18.659691
2026-03-09 08:00:00,17.895618
2026-03-09 08:15:00,17.131545



=== REPORT: SOIL_UMID_PCT ===
Acceptable Limits: 27.63 to 52.04
Anomalies found: 0
[v] No critical errors found.
